# 01 — Offline environment setup

Checks that this checkout can run the sandbox **offline**, before any data prep or eval.

| Step | What it does | Writes |
| --- | --- | --- |
| 1. Bootstrap | Finds the checkout and pins an absolute data directory | nothing |
| 2. Versions | Python, the sandbox package, notebook libraries and vendored snapshots | nothing |
| 3. Dotenv | Copies `config/.env.example` to `.env` **only if `.env` is missing** | `.env` (gitignored) |
| 4. Profile + checklist | Activates the offline profile and checks the Dockerfile, Compose file and fixtures | `data/runtime/taxonomy.yaml` |
| 5. Provider probe (opt-in) | Calls the local model endpoint for the profile | nothing |

**Cost:** none. No step launches paid or remote work. The provider probe in step 5 is off by default and only calls local endpoints such as Ollama.

**Order:** run this notebook first, then [`02_load_clean_prepare_data`](02_load_clean_prepare_data.ipynb), then [`03_offline_sandbox_smoke`](03_offline_sandbox_smoke.ipynb). [`04_experiment_log_explorer`](04_experiment_log_explorer.ipynb) is optional.

## 1. Bootstrap

`mailroom_sandbox.notebook.bootstrap()` is the shared first step for every notebook. It:

- finds the checkout from the kernel's working directory. Jupyter starts kernels in `notebooks/`, not at the repo root.
- sets `SANDBOX_ROOT` and an **absolute** `MAILROOM_BASE_DIR`. If the relative `./data` from `.env` were left as is, the run store would end up in `notebooks/data/`.
- uses the `ollama` profile unless the kernel was started with `SANDBOX_PROFILE` set (Compose's `jupyter` service sets it). The `modal-vllm` default in `.env` is ignored on purpose.
- turns tracing off (`OBSERVABILITY_PROVIDER=none`) and lowers pipeline logging to `WARNING`.

This notebook runs the dotenv and activation steps one at a time below, so here bootstrap only resolves paths.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Works with or without `pip install -e .`: fall back to the checkout's src/ when needed.
try:
    import mailroom_sandbox  # noqa: F401
except ModuleNotFoundError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "src" / "mailroom_sandbox").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import mailroom_sandbox  # noqa: F401

import os

from mailroom_sandbox.notebook import bootstrap

ctx = bootstrap(create_dotenv=False, activate_profile=False)
print(json.dumps(ctx.as_dict(), indent=2))

{
  "root": "/demo/checkout",
  "profile": "ollama",
  "mock": true,
  "observability": "none",
  "data_dir": "/demo/runtime/data",
  "prepared_dir": "/demo/runtime/data/runtime/prepared",
  "experiment_log": "/demo/runtime/log/experiment_log.jsonl"
}


## 2. Interpreter, packages and vendored snapshots

The sandbox imports the family pipeline and scoring from **tracked snapshots** under `vendor/`. Never hand-edit them. Use `python scripts/sync_vendor.py` from the owning repos instead. Missing notebook libraries mean the `[notebooks]` extra is not installed.

In [2]:
import platform
from importlib import metadata


def version(dist: str) -> str:
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return "not installed"


print(f"python      {platform.python_version()}  ({sys.executable})")
for dist in ("mailroom-sandbox", "jupyterlab", "ipykernel", "pandas", "matplotlib", "langgraph"):
    print(f"{dist:<12}{version(dist)}")

assert sys.version_info >= (3, 11), "the sandbox needs Python 3.11+"

print()
for name in ("llm-mailroom", "llm-dojo-scoring"):
    path = ctx.root / "vendor" / name
    print(f"vendor/{name:<17}{'present' if path.is_dir() else 'MISSING'}")

python      3.13.16  (/demo/python)
mailroom-sandbox0.3.0
jupyterlab  not installed
ipykernel   7.4.0
pandas      3.1.1
matplotlib  3.11.2
langgraph   1.2.22

vendor/llm-mailroom     present
vendor/llm-dojo-scoring present


## 3. Dotenv

`ensure_dotenv_from_example()` **never overwrites** an existing `.env`. It only creates one from `config/.env.example` when none exists. Secrets belong in `.env`, which is gitignored. This notebook shows only whether the file exists and a few non-secret settings, never values that could be credentials.

Inside Compose, the `jupyter` service injects sibling URLs (`http://ollama:11434/v1`, `http://langfuse-web:3000`). On the host, the `localhost` defaults from the example apply.

In [3]:
from mailroom_sandbox.prep import ensure_dotenv_from_example

env_path = ensure_dotenv_from_example()
assert env_path is not None and env_path.is_file(), "config/.env.example is missing; cannot create .env"
print("dotenv →", env_path.relative_to(ctx.root))

NON_SECRET = ("SANDBOX_PROFILE", "MAILROOM_BASE_DIR", "OBSERVABILITY_PROVIDER")
from dotenv import dotenv_values

file_values = {} if os.environ.get("SANDBOX_SKIP_DOTENV") else dotenv_values(env_path)
for key in NON_SECRET:
    print(f"  .env {key:<24}= {file_values.get(key)!r}")
print(f"\nnotebook profile = {ctx.profile!r} (the .env SANDBOX_PROFILE is not used here)")

dotenv → .env
  .env SANDBOX_PROFILE         = None
  .env MAILROOM_BASE_DIR       = None
  .env OBSERVABILITY_PROVIDER  = None

notebook profile = 'ollama' (the .env SANDBOX_PROFILE is not used here)


## 4. Activate the profile and run the checklist

`environment_checklist()` activates the profile, which writes `data/runtime/taxonomy.yaml`, then checks the files the offline image and Compose stack need. `offline_ready` is true only when all of these hold: fixtures, Compose file, Dockerfile, a successful activation and a dotenv file.

In [4]:
try:
    import pandas as pd
except ImportError as exc:  # pragma: no cover - guidance for a bare [dev] install
    raise ImportError('pandas is missing: pip install -e ".[dev,notebooks]"') from exc

pd.set_option("display.max_colwidth", 80)

from mailroom_sandbox.prep import environment_checklist

check = environment_checklist(ctx.profile)
summary = {k: v for k, v in check.items() if k not in ("profiles_available", "notebooks")}
display(pd.Series(summary, name="value").to_frame())
print("profiles available:", ", ".join(check["profiles_available"]))
print("notebooks found:   ", ", ".join(Path(p).name for p in check["notebooks"]))

,value
repo_root,/demo/checkout
profile,ollama
env_file,.env
fixtures_manifest_ok,True
compose_file_ok,True
dockerfile_ok,True
mailroom_src,/demo/checkout/vendor/llm-mailroom/src
runtime_taxonomy,/demo/runtime/data/runtime/taxonomy.yaml
agent_count,16
activation_ok,True


profiles available: llamacpp, lmstudio, modal-vllm, ollama, openrouter, vllm-local, vllm-remote
notebooks found:    01_offline_environment_setup.ipynb, 02_load_clean_prepare_data.ipynb, 03_offline_sandbox_smoke.ipynb, 04_experiment_log_explorer.ipynb


In [5]:
failures = {
    "deploy/Dockerfile missing": not check["dockerfile_ok"],
    "deploy/docker-compose.yml missing": not check["compose_file_ok"],
    "data/fixtures/manifest.csv missing": not check["fixtures_manifest_ok"],
    f"profile {ctx.profile!r} did not activate: {check['activation_error']}": not check["activation_ok"],
    "no notebooks found under notebooks/": not check["notebooks"],
}
problems = [msg for msg, failed in failures.items() if failed]
assert not problems, "\n".join(problems)
print(f"offline_ready = {check['offline_ready']}  ({check['agent_count']} agents assigned by {ctx.profile!r})")

offline_ready = True  (16 agents assigned by 'ollama')


## 5. Optional: probe the local provider

Off by default, because notebooks 02 and 03 need no model server. To run it, start the provider first, then set `PROBE = True` here or start Jupyter with `SANDBOX_NOTEBOOK_PROBE=1`. For Ollama, run `sandbox up --compose-profile ollama` and `sandbox pull-models --profile ollama`. The probe calls the profile's own endpoints, which for `ollama` means a local server.

In [6]:
PROBE = os.environ.get("SANDBOX_NOTEBOOK_PROBE") == "1"

if PROBE:
    from mailroom_sandbox.health import health_check
    from mailroom_sandbox.notebook import probe_refusal

    try:
        # health_check sends a real chat completion: only a local endpoint is probed from here.
        refusal = probe_refusal(ctx.profile)
        if refusal:
            print(f"provider probe refused: {refusal}; run `sandbox health --profile {ctx.profile}` instead")
        else:
            print(json.dumps(health_check(ctx.profile, cloud_diagnostics=False), indent=2, default=str))
    except Exception as exc:  # noqa: BLE001 - report, don't fail the setup notebook
        print(f"probe failed: {type(exc).__name__}: {exc}")
else:
    print("provider probe skipped (set SANDBOX_NOTEBOOK_PROBE=1 to run it)")

provider probe skipped (set SANDBOX_NOTEBOOK_PROBE=1 to run it)


## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `ModuleNotFoundError: mailroom_sandbox` | `pip install -e ".[dev,notebooks]"` from the repo root, then pick that interpreter's kernel |
| `ModuleNotFoundError: pandas` | install the `[notebooks]` extra |
| `activation_ok` is false | read `activation_error`. A profile without a local endpoint, such as `modal-vllm` with no `VLLM_BASE_URL`, cannot be used offline, so keep `ollama` |
| A `notebooks/data/` folder appeared | it came from an older notebook version. Delete it. The bootstrap now pins `MAILROOM_BASE_DIR` |

**Next:** [`02_load_clean_prepare_data.ipynb`](02_load_clean_prepare_data.ipynb).